# Image retrieval: what does each kind of pretraining teach a model to find?

**Goal:** given a photo, find the other photos of the same thing in a small collection.
We compare six **frozen** pretrained vision models (**ViT, ConvNeXt, DINOv2, I-JEPA,
V-JEPA 2, and DiT**) without training anything. Each model turns every image into one
vector; we rank the collection by cosine similarity and check whether the right images
come first.

The lesson asks four questions:
1. Does the pretraining objective (labels, matching views, predicting hidden regions, or
   removing noise) change what a model can retrieve?
2. Does one model win on every kind of image, or does the winner depend on the domain?
3. Which photo edits does each model ignore, and is ignoring an edit always good?
4. Where inside each network does the most useful information live?

**You only need to run this notebook.** From `homework_2/`:
```bash
uv sync --locked
uv run jupyter lab vision_ir/teaching.ipynb
```
Then choose **Run → Run All Cells**. The first run downloads about 150 MB of images and
7.6 GB of model weights, then embeds 2,250 images with each model: about 10 minutes on a
recent Apple-silicon Mac or a PACE ICE GPU, longer on smaller laptops (see the README,
which also has a PACE batch job). Everything is cached, so running it again takes about a minute.

The supporting code sits next to this notebook: [data](data.py), [models](models.py),
[embedding cache](embed.py), [photo edits](augment.py), [metrics](evaluate.py),
[tasks](tasks.py), and [plots](plots.py).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "vision_ir/models.py").is_file())
sys.path.insert(0, str(ROOT))

from vision_ir.augment import AUGMENTATIONS, augment
from vision_ir.data import DOMAINS, copy_queries, gallery_items, prepare_gpr1200, read_photos
from vision_ir.embed import embed
from vision_ir.evaluate import average_precision, cosine_scores, mean_pair_cosine, standardize, top_k
from vision_ir.models import BACKBONES, select_device
from vision_ir.paths import DATA, OUTPUT, model_cache
from vision_ir.plots import MODEL_COLORS, heatmap, layer_curves, show_domains, show_rankings
from vision_ir.tasks import copy_scores, same_group_per_query, same_group_scores

CACHE = model_cache()           # Hugging Face weights (respects HF_HOME, e.g. on PACE scratch)
EMBEDDINGS = OUTPUT / "embeddings"
OUTPUT.mkdir(parents=True, exist_ok=True)
device = select_device()
print(f"Device: {device}\nModel weights: {CACHE}\nResults: {OUTPUT}")

## 0. Download the images

We use a fixed slice of **GPR1200** ([Schall et al., 2021](https://arxiv.org/abs/2111.13122)),
a benchmark built for *general-purpose* image retrieval. It has 1,200 **groups** of 10 images,
and each block of 200 groups comes from a different source. We keep every 8th group from
five blocks: **125 groups × 10 images = 1,250 images**. The sixth block (celebrity faces)
is skipped on purpose.

The cell reads only those 1,250 JPEGs out of the 1.25 GB archive (using HTTP byte-range
requests) and stores them in `data/gpr1200/`. Later runs reuse the files. GPR1200 is provided
for non-commercial research and educational use only; the images belong to their owners.

In [ ]:
photos = read_photos(prepare_gpr1200(DATA))
copies = copy_queries(photos)
gallery_ids = [photo.image_id for photo in photos]
copy_ids = [copy.item.item_id for copy in copies]
print(f"{len(photos):,} gallery images in {len({p.group for p in photos})} groups; "
      f"{len(copies):,} edited copies for Task 2")

## 1. Five kinds of "the same"

| Domain | Source | Two images share a group when they show… | Kind |
|---|---|---|---|
| Landmarks | Google Landmarks v2 | the same building or place | instance |
| Nature | iNaturalist | the same species | category |
| Sketches | ImageNet-Sketch | the same ImageNet class, drawn by hand | category |
| Objects | INSTRE | the same physical object | instance |
| Products | Stanford Online Products | the same product for sale | instance |

**Instance** retrieval asks for *the same thing* (this exact bridge). **Category** retrieval
asks for *the same kind of thing* (another photo of this species). Every search covers the
whole collection, so a query competes with all 1,249 other images from every domain.

In [ ]:
show_domains(photos)
plt.show()

## 2. Two tasks and two metrics

**Task 1, find the same thing.** Every gallery image is used as a query, as in GPR1200. Unlike
GPR1200's official script, we remove the query from its own results, so the 9 other images
of its group are the correct answers.

**Task 2, find the original.** We take the first image of each group and edit it in 8 ways
(below), giving 1,000 queries. Exactly one image is correct: the unedited original. Its 9
group-mates count as *wrong*, so a model must recognize this exact photo, not merely the
same object.

| Metric | Question it answers |
|---|---|
| **P@1** | Is the top-ranked image correct? |
| **mAP** | For each query, take the precision at the rank of every correct image and average them (AP); then average over queries. mAP = 1 means every correct image is ranked above every wrong one. |

With correct images at ranks 1 and 3, AP = (1/1 + 2/3) / 2 = 0.833. When exactly one image is
correct, AP equals 1 / its rank, the reciprocal rank behind Part 1's MRR. Both metrics range
from 0 to 1; higher is better.

In [ ]:
print("Correct images at ranks 1 and 3 → AP =", round(average_precision([True, False, True, False]), 3))

original = Image.open(photos[0].path).convert("RGB")
examples = [("original", original)] + [(name, augment(original, name)) for name in AUGMENTATIONS]
figure, axes = plt.subplots(1, len(examples), figsize=(1.9 * len(examples), 2.3))
for axis, (name, image) in zip(axes, examples, strict=True):
    axis.imshow(image)
    axis.set_title(name, fontsize=9)
    axis.axis("off")
figure.suptitle("The eight edits used to build Task 2 queries")
plt.show()

## 3. Six pretrained models

| Model | Size | Learns from | Training signal | Vector we use |
|---|---|---|---|---|
| [**ViT-B/16**](https://arxiv.org/abs/2010.11929) | 86M | ImageNet-21k → 1k **labels** | name the class of each image | CLS token |
| [**ConvNeXt-B**](https://arxiv.org/abs/2201.03545) | 88M | ImageNet-22k → 1k **labels** | same task, but a CNN instead of a Transformer | average of the final feature map |
| [**DINOv2 ViT-B/14**](https://arxiv.org/abs/2304.07193) | 87M | 142M images, no labels | match a large DINOv2 teacher on differently cropped, flipped, recolored, and blurred views | CLS token |
| [**I-JEPA ViT-H/14**](https://arxiv.org/abs/2301.08243) | 631M | ImageNet-1k images, no labels | predict the *embeddings* of hidden image blocks from a visible block | average of patch tokens |
| [**V-JEPA 2 ViT-L/16**](https://arxiv.org/abs/2506.09985) | 304M | ~21M videos + 1M ImageNet images, no labels | the same idea for video: predict embeddings of hidden space-time regions | average of patch tokens |
| [**DiT-XL/2**](https://arxiv.org/abs/2212.09748) | 709M | ImageNet-1k images + labels | remove noise so it can *generate* images | average of block-14 tokens |

A Transformer cuts the image into patches and turns each one into a **token**. The **CLS
token** is an extra token whose output summarizes the whole image; models without one are
summarized by averaging their patch tokens.

**JEPA** stands for *Joint-Embedding Predictive Architecture*: instead of predicting pixels,
the model predicts the representation of what is hidden. I-JEPA does this within one image;
V-JEPA 2 does it across video frames, so we feed each photo as a two-frame still clip.
LeCun's [proposed *hierarchical* JEPA](https://openreview.net/pdf?id=BZ5a1r-kVsf) (H-JEPA)
has, as of November 2026, been released only as [robot-planning world models](https://arxiv.org/abs/2610.06805),
not as a general image encoder, so V-JEPA 2 is our second JEPA. (The newer
[V-JEPA 2.1](https://arxiv.org/abs/2603.14482) also handles images, but it is only available
through `torch.hub`, not the pinned Hugging Face checkpoints used here.)

**DiT** never learned to produce an embedding. It works on compressed images: a **VAE**
squeezes each 256 px photo into a small grid of numbers. We add a little noise (step *t* = 100
of 1,000), run one denoising pass with the "no class" label, and average the tokens leaving
block 14 of 28, following [DDAE (Xiang et al., 2023)](https://arxiv.org/abs/2303.09769).
This is the original 2023 DiT, trained only to denoise. Newer diffusion transformers often align
their middle layers with DINOv2-like features ([REPA](https://arxiv.org/abs/2410.06940)) or generate
directly in a DINOv2 feature space ([RAE](https://arxiv.org/abs/2510.11690)); their features would
score much higher here.

Every model sees the same center crop (224 px, or 256 px for V-JEPA 2 and DiT), normalized
the way it was trained. The public I-JEPA weights are not the exact copy its paper evaluated,
so treat its score as a lower bound. This is a **practical comparison**, not a controlled
experiment: the models differ in size, training data, and resolution. DINOv2's ViT-B is a small
model distilled from a much larger teacher, and [DINOv3](https://arxiv.org/abs/2508.10104) (2025)
is its successor; we use DINOv2 because its weights need no sign-up.

## 4. Embed every image once

Each model embeds the 1,250 gallery images and the 1,000 edited copies (2,250 images).
Vectors are saved in `artifacts/vision/embeddings/`, so later runs load them instantly.
Only one model is in memory at a time. The time shown is how long that model took on the
run that created its cache.

In [ ]:
items = gallery_items(photos) + [copy.item for copy in copies]
embeddings = {}
for key in BACKBONES:
    embeddings[key] = embed(key, items, cache_dir=CACHE, output_dir=EMBEDDINGS, device=device)
    e = embeddings[key]
    print(f"{key:9s} {e.final.shape[1]:5d}-dimensional vectors   {e.seconds:6.0f} s on {e.device}")

## 5. Task 1: find the same thing

The first table averages over all 1,250 queries. The heatmap splits mAP by the domain of
the query. Look for:

- the overall ranking, and how far apart the six models are;
- whether the same model wins every column;
- which domains are easy for every model and which separate them.

With only 125 groups, differences of about 0.01 (for example ConvNeXt versus DINOv2) are within
noise; look for larger gaps.

In [ ]:
task1 = pd.DataFrame({key: same_group_scores(e.rows(gallery_ids), photos) for key, e in embeddings.items()}).T
task1.index.name = "model"
domain_names = [domain.name for domain in DOMAINS]
display(task1[["mAP", "P@1"]].style.format("{:.3f}").highlight_max(color="#cfe8cf"))
heatmap(task1[domain_names], "Task 1 mAP by query domain")
plt.show()
task1.to_csv(OUTPUT / "task1_same_group.csv")

### What the rankings look like

For three domains we pick the query on which the six models **disagree most** (largest gap
between the best and worst AP), then show each model's top five results. Green borders
mark images from the query's group; orange borders mark mistakes. Ask yourself what the
mistakes have in common with the query: color, background, layout, or meaning?

In [ ]:
per_query = pd.DataFrame({key: same_group_per_query(e.rows(gallery_ids), photos)["mAP"]
                          for key, e in embeddings.items()})
domains = np.array([photo.domain for photo in photos])
for domain in ("Landmarks", "Nature", "Objects"):
    rows = per_query[domains == domain]
    query = int((rows.max(axis=1) - rows.min(axis=1)).idxmax())
    ranked = {}
    for key, e in embeddings.items():
        vectors = e.rows(gallery_ids)
        scores = cosine_scores(vectors[query : query + 1], vectors)
        ignore = np.zeros_like(scores, dtype=bool)
        ignore[0, query] = True  # do not return the query itself
        ranked[key] = [photos[i] for i in top_k(scores, 5, ignore)[0]]
    show_rankings(photos[query], ranked, f"{domain}: green = same group, orange = mistake")
    plt.show()

## 6. Task 2: find the original after an edit

Each heatmap cell is the fraction of 125 edited copies whose top result is the exact
original. Compare the columns with what each model saw as a training augmentation:

| Edit | Seen during pretraining by |
|---|---|
| flip | ViT, ConvNeXt, DINOv2, V-JEPA 2, DiT (I-JEPA's training turns flips off) |
| crop | random crops: all except DiT, which trained on a fixed center crop |
| grayscale, hue shift | DINOv2 (random grayscale, mild hue jitter); ConvNeXt only partly desaturates, never fully grays or shifts hue |
| blur | DINOv2 |
| JPEG, pixelate, rotate 90° | none explicitly (ConvNeXt's random rotations stay within 30°) |

The table below the heatmap adds one more column. When a model misses, how often is its top
result one of the original's 9 **group-mates**? A high share means the model recognized the
object but could not tell this exact photo from other photos of it.

In [ ]:
task2 = pd.DataFrame({key: copy_scores(e.rows(copy_ids), e.rows(gallery_ids), photos, copies)
                      for key, e in embeddings.items()}).T
task2.index.name = "model"
heatmap(task2[list(AUGMENTATIONS)], "Task 2 P@1: is the unedited original the top result?")
plt.show()
display(task2[["mean", "misses that were group-mates"]].rename(columns={"mean": "mean P@1"})
        .style.format("{:.3f}"))
task2.to_csv(OUTPUT / "task2_find_original.csv")

## 7. Where inside the network?

Networks build features in stages, from local edges and textures toward whole objects;
H-JEPA's levels of abstraction are a loose cousin of this idea. During embedding we also saved
the **average of the patch tokens leaving 8 evenly spaced blocks** of each Transformer (the 4
stages of ConvNeXt). Each curve shows Task 1 mAP when we retrieve with features from that depth.

The curves always average patch tokens, so their last point can sit below Section 5, which
used the CLS token for ViT and DINOv2. DINOv2 applies its whole-image training loss to the CLS
token and trains its patch tokens for local detail, so for DINOv2 the gap is expected.

In [ ]:
layer_rows = []
for key, e in embeddings.items():
    deepest = max(e.layers)
    for block in sorted(e.layers):
        vectors = e.layer_rows(block, gallery_ids).astype(np.float32)
        layer_rows.append({"model": key, "block": block, "depth": block / deepest,
                           "mAP": same_group_scores(vectors, photos)["mAP"]})
layers = pd.DataFrame(layer_rows)
layer_curves(layers, "Task 1 mAP using the average patch token at each depth")
plt.show()

best = layers.loc[layers.groupby("model", sort=False)["mAP"].idxmax()].set_index("model")
display(pd.DataFrame({
    "best block": best["block"],
    "of": [max(e.layers) for e in embeddings.values()],
    "mAP at best block": best["mAP"],
    "mAP at last block": layers.groupby("model", sort=False)["mAP"].last(),
    "Section 5 mAP": task1["mAP"],
}).style.format({c: "{:.3f}" for c in ["mAP at best block", "mAP at last block", "Section 5 mAP"]}))

## 8. Why are some vectors harder to search?

If every vector points in nearly the same direction, all images look alike to cosine
similarity. The first column is the **average cosine similarity between two different
images**: near 0 means vectors are spread out, near 1 means they are crowded together.

A common fix needs no labels: **standardize** each dimension (subtract its mean over the
gallery and divide by its standard deviation) before comparing. This is a simple, per-dimension
form of *whitening*; image-retrieval systems often use full PCA whitening, which we do not try here. The last column repeats
Task 1 after that fix.

In [ ]:
crowding = pd.DataFrame({
    "average cosine between different images": {
        k: mean_pair_cosine(e.rows(gallery_ids)) for k, e in embeddings.items()
    },
    "Task 1 mAP": task1["mAP"],
    "Task 1 mAP, standardized": {
        k: same_group_scores(standardize(e.rows(gallery_ids), e.rows(gallery_ids)), photos)["mAP"]
        for k, e in embeddings.items()
    },
})
display(crowding.style.format("{:.3f}"))

## 9. Summary

In [ ]:
summary = pd.DataFrame({
    "parameters (M)": {key: round(b.params_m) for key, b in BACKBONES.items()},
    "embedding time (s)": {key: round(e.seconds) for key, e in embeddings.items()},
    "Task 1 mAP": task1["mAP"],
    "Task 1 P@1": task1["P@1"],
    "Task 2 P@1": task2["mean"],
    "Task 1 mAP, standardized": crowding["Task 1 mAP, standardized"],
})
summary.index.name = "model"
display(summary.style.format({c: "{:.3f}" for c in summary.columns if "mAP" in c or "P@1" in c}))
summary.to_csv(OUTPUT / "summary.csv")

figure, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, layout="constrained")
for axis, column in zip(axes, ["Task 1 mAP", "Task 2 P@1"], strict=True):
    bars = axis.barh(summary.index, summary[column], color=[MODEL_COLORS[m] for m in summary.index])
    axis.bar_label(bars, fmt="%.3f", padding=3)
    axis.set(title=column, xlim=(0, 1.12))
axes[0].invert_yaxis()
figure.suptitle("Same 1,250 images, six frozen models")
figure.savefig(OUTPUT / "summary.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. What should we learn from the comparison?

- **The training signal shapes the embedding.** Models trained with labels (ViT, ConvNeXt) or
  to match views of the same image (DINOv2) give retrieval-ready vectors. Picking a better layer
  or standardizing does not close the gap for the JEPA and diffusion models.
- **"The same" changes which models look good.** Landmarks and Products are the easiest domains
  for every model; Nature, Sketches, and Objects separate them. The top two models trade wins by domain.
- **Invariance is a choice.** Every model struggles with edits it never trained on (pixelate,
  90° rotation; blur for V-JEPA 2). When the strongest models miss, they usually return a group-mate: they
  recognized the object but not this photo. The weaker models usually return something unrelated.
- **Layer and pooling both matter.** DiT's best features sit in the middle of the network and
  the JEPA models peak just before their last block. For DINOv2, the CLS token beats the average
  of its patch tokens by a wide margin.
- **Report your post-processing.** Standardizing helps crowded vectors most, but here it changes
  no model's rank: this simple fix does not close DiT's gap (full PCA whitening was not tried).

**Discuss** (answer in `homework_2/README.md`, Part 2):

1. Which domains are easy for every model, and which separate them? Look at the example rankings: what did the wrong results share with the query (color, background, layout, or meaning)?

1. ViT and ConvNeXt were trained to predict ImageNet labels, DiT was given them as an input while learning to denoise, and the Sketches domain uses ImageNet classes. DINOv2, I-JEPA, and V-JEPA 2 also trained on ImageNet images (without labels), and DINOv2's training set was assembled partly by similarity to retrieval datasets such as Google Landmarks. Does the Sketches column show an advantage for the label-trained models? How could these overlaps affect the comparison?

1. In Task 2, name one result that the augmentation table predicts and one that it does not. When ViT, ConvNeXt, or DINOv2 miss, what do they usually return instead, and how is that different for the JEPA models and DiT?

1. Where does each depth curve peak? Why might a model trained to denoise or to predict hidden regions not put its best features in its last layer? ViT's and DINOv2's last points sit below their Task 1 scores: what does that say about pooling (CLS token versus the average of patch tokens)?

1. DiT's raw vectors have an average cosine near 1, yet standardizing changed no model's rank. What does that say about why DiT trails? I-JEPA and DiT are 7–8× larger than DINOv2: using the embedding times in the summary, what did the extra parameters buy?

These results come from a 1,250-image slice with one fixed setting per model, so compare patterns across domains and edits rather than small differences.